# VT-Guard: baseline CPU trên Colab
Notebook chỉ điều phối module trong repository. Điền Git URL và commit cố định. Chạy smoke tổng hợp trước; chỉ chạy dữ liệu thật khi manifest đã có run và annotation được duyệt. Lưu manifest, raw có quyền sử dụng và artifact trên Drive. Không nhập credential vào notebook.

In [ ]:
import subprocess, sys
from pathlib import Path
REPO_URL = ''  # Điền GitHub URL sau khi repo được tạo
CODE_COMMIT = ''  # Full commit SHA, không dùng nhánh di động
if not REPO_URL or len(CODE_COMMIT) != 40:
    raise ValueError('Cần GitHub URL và full commit SHA trước khi dùng Colab')
repo = Path('/content/vt-guard')
subprocess.run(['git', 'clone', '--filter=blob:none', REPO_URL, str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', CODE_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(repo)], check=True)
print('Python:', sys.version)


In [ ]:
# Kiểm thử nhanh chỉ với dữ liệu tổng hợp; kết quả không là điểm phát hiện.
subprocess.run([sys.executable, str(repo / 'scripts/smoke_baseline.py')], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
MANIFEST = Path('/content/drive/MyDrive/vtguard/reviewed_manifest.json')
OUTPUT = Path('/content/drive/MyDrive/vtguard/baseline_v1')
if not MANIFEST.is_file():
    raise FileNotFoundError('Thiếu reviewed manifest và các tệp raw tương ứng trên Drive')
subprocess.run([sys.executable, str(repo / 'scripts/train_baseline.py'),
                '--manifest', str(MANIFEST), '--output', str(OUTPUT)], check=True)
